In [95]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

incoming data:
x,y,z,a,b,c,timestamp

In [121]:
df_phone = pd.read_csv("../data/test/test_phone.csv")
df_watch = pd.read_csv("../data/test/test_watch.csv")
df_aligned = pd.read_csv("../data/test/test_aligned(8s).csv")

In [122]:
df_phone["timeStamp"] = pd.to_datetime(df_phone["timeStamp"], unit="ns")
df_watch["timeStamp"] = pd.to_datetime(df_watch["timeStamp"], unit="ns")
df_aligned["rel_time"] = pd.to_timedelta(df_aligned["rel_time"])

In [127]:
df_phone["timeStamp"] = pd.to_datetime(df_phone["timeStamp"], unit="ns")
df_phone

,timeStamp,x_accel,y_accel,z_accel,x_gyro,y_gyro,z_gyro
0,1970-01-01 00:23:52.758153112,-0.530609,-9.637375,-0.548340,-0.044083,0.233490,0.112076
1,1970-01-01 00:23:52.808507116,-0.805191,-9.644119,-0.906708,0.016617,0.274658,0.067566
2,1970-01-01 00:23:52.858861120,0.368484,-9.754166,-1.373718,0.071365,0.317673,0.028641
3,1970-01-01 00:23:52.909215124,0.302917,-9.567200,-1.122131,-0.001389,-0.133652,0.064774
4,1970-01-01 00:23:52.959569128,-1.290131,-9.871155,-0.356186,-0.037842,-0.434036,0.080750
...,...,...,...,...,...,...,...
75,1970-01-01 00:23:56.534703405,-1.310822,-9.655899,-1.001175,0.021820,-0.031769,0.001785
76,1970-01-01 00:23:56.585057409,-1.588150,-9.682159,-0.865387,-0.003113,-0.023346,-0.008743
77,1970-01-01 00:23:56.635411413,-1.313049,-9.707428,-0.668854,-0.026566,0.040359,-0.012787
78,1970-01-01 00:23:56.685765417,-1.159561,-9.581451,-0.739883,0.005798,0.035614,-0.001022


In [124]:
import json

df_phone["timeStamp"] = df_phone["timeStamp"].astype("int64")
records = df_phone.to_dict(orient="records")
output = {"data": records}

with open("../data/test/test_phone(4s).json", "w") as f:
    json.dump(output, f, indent=2)

In [102]:
df_phone.T.to_json("../data/test/test_phone(4s).json",index=False)

In [84]:
df_phone["rel_time"] = df_phone["timeStamp"] - df_phone["timeStamp"].min()
df_phone = df_phone.drop(columns="timeStamp")

df_watch["rel_time"] = df_watch["timeStamp"] - df_watch["timeStamp"].min()
df_watch = df_watch.drop(columns="timeStamp")

df_aligned = df_aligned.drop(columns="subject_id")

In [85]:
def extract_windows(df, window_sec=2):
    df = df.copy()
    df = df.sort_values(["rel_time"])
    df = df.set_index("rel_time")

    sensor_cols = ["x_accel", "y_accel", "z_accel", "x_gyro", "y_gyro", "z_gyro"]
    fs = 20.0  # sampling frequency

    grouped = df.groupby(
        [pd.Grouper(freq=f"{window_sec}s")],
        observed=True,
    )

    windows = []
    for name, group in grouped:
        window_start = name
        features = {
            "window_start": window_start,
        }

        for axis in sensor_cols:
            series = group[axis].values
            n = len(series)
            if n == 0:
                continue

            # Time domain features
            mean_val = np.mean(series)
            std_val = np.std(series, ddof=1) if n > 1 else np.nan
            p25 = np.percentile(series, 25)
            p75 = np.percentile(series, 75)
            kurt = pd.Series(series).kurtosis()

            with np.errstate(invalid="ignore", divide="ignore"):
                autocorr = pd.Series(series).autocorr(lag=1) if n > 2 else np.nan

            rms = np.sqrt(np.mean(np.square(series)))

            # Frequency domain features
            if n > 1:
                fft_vals = np.fft.rfft(series)
                freqs = np.fft.rfftfreq(n, d=1 / fs)
                magnitudes = np.abs(fft_vals)
                if len(magnitudes) > 1:
                    idx_max = np.argmax(magnitudes[1:]) + 1
                    dom_freq = freqs[idx_max]
                    spectral_energy = np.sum(magnitudes[1:] ** 2)
                else:
                    dom_freq = 0.0
                    spectral_energy = 0.0
            else:
                dom_freq = 0.0
                spectral_energy = 0.0

            prefix = axis + "_"
            features[prefix + "mean"] = mean_val
            features[prefix + "std"] = std_val
            features[prefix + "p25"] = p25
            features[prefix + "p75"] = p75
            features[prefix + "kurt"] = kurt
            features[prefix + "autocorr"] = autocorr
            features[prefix + "rms"] = rms
            features[prefix + "dom_freq"] = dom_freq
            features[prefix + "spectral_energy"] = spectral_energy

        windows.append(features)

    windows_df = pd.DataFrame(windows)
    return windows_df.dropna().drop(columns="window_start")

In [86]:
def extract_windows_aligned(df, window_sec=2):
    df = df.copy()
    df = df.sort_values(["rel_time"])
    df = df.set_index("rel_time")

    sensor_cols = [
        "x_phone_accel",
        "y_phone_accel",
        "z_phone_accel",
        "x_phone_gyro",
        "y_phone_gyro",
        "z_phone_gyro",
        "x_watch_accel",
        "y_watch_accel",
        "z_watch_accel",
        "x_watch_gyro",
        "y_watch_gyro",
        "z_watch_gyro",
    ]
    fs = 20.0

    grouped = df.groupby(
        [pd.Grouper(freq=f"{window_sec}s")],
        observed=True,
    )

    windows = []
    for name, group in grouped:
        window_start = name

        features = {"window_start": window_start}

        for axis in sensor_cols:
            series = group[axis].values
            n = len(series)
            if n == 0:
                continue
            mean_val = np.mean(series)
            std_val = np.std(series, ddof=1) if n > 1 else np.nan
            p25 = np.percentile(series, 25)
            p75 = np.percentile(series, 75)
            kurt = pd.Series(series).kurtosis()

            with np.errstate(invalid="ignore", divide="ignore"):
                autocorr = pd.Series(series).autocorr(lag=1) if n > 2 else np.nan

            rms = np.sqrt(np.mean(np.square(series)))

            if n > 1:
                fft_vals = np.fft.rfft(series)
                freqs = np.fft.rfftfreq(n, d=1 / fs)
                magnitudes = np.abs(fft_vals)
                if len(magnitudes) > 1:
                    idx_max = np.argmax(magnitudes[1:]) + 1
                    dom_freq = freqs[idx_max]
                    spectral_energy = np.sum(magnitudes[1:] ** 2)
                else:
                    dom_freq = 0.0
                    spectral_energy = 0.0
            else:
                dom_freq = 0.0
                spectral_energy = 0.0

            prefix = axis + "_"
            features[prefix + "mean"] = mean_val
            features[prefix + "std"] = std_val
            features[prefix + "p25"] = p25
            features[prefix + "p75"] = p75
            features[prefix + "kurt"] = kurt
            features[prefix + "autocorr"] = autocorr
            features[prefix + "rms"] = rms
            features[prefix + "dom_freq"] = dom_freq
            features[prefix + "spectral_energy"] = spectral_energy

        windows.append(features)

    windows_df = pd.DataFrame(windows)
    windows_df = windows_df.reset_index(drop=True)
    return windows_df.dropna().drop(columns="window_start")

In [87]:
extract_windows_aligned(df_aligned, window_sec=4)

,x_phone_accel_mean,x_phone_accel_std,x_phone_accel_p25,x_phone_accel_p75,x_phone_accel_kurt,x_phone_accel_autocorr,x_phone_accel_rms,x_phone_accel_dom_freq,x_phone_accel_spectral_energy,y_phone_accel_mean,...,y_watch_gyro_spectral_energy,z_watch_gyro_mean,z_watch_gyro_std,z_watch_gyro_p25,z_watch_gyro_p75,z_watch_gyro_kurt,z_watch_gyro_autocorr,z_watch_gyro_rms,z_watch_gyro_dom_freq,z_watch_gyro_spectral_energy
0,1.387719,0.040473,1.365032,1.416756,-0.213012,0.654660,1.388301,0.250000,5.177311,6.209856,...,121.317040,0.131556,0.339942,-0.029565,0.266846,0.789788,0.958187,0.362523,0.250000,365.171620
1,1.448764,0.052086,1.408432,1.476311,0.141612,0.474184,1.449688,0.253165,8.358474,6.274451,...,61.153418,-0.019230,0.495199,-0.417855,0.261783,-0.846374,0.981624,0.492430,0.253165,755.527614


In [88]:
from sklearn.preprocessing import FunctionTransformer

window_extractor = FunctionTransformer(extract_windows, kw_args={"window_sec": 4})
window_extractor_aligned = FunctionTransformer(
    extract_windows_aligned, kw_args={"window_sec": 4}
)

In [89]:
from sklearn.base import BaseEstimator, TransformerMixin

class RawFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, mode="phone", window_sec=4):
        self.mode = mode
        self.window_sec = window_sec

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if X is None:
            return X

        frame = X.copy()
        if "rel_time" not in frame.columns and "timeStamp" in frame.columns:
            frame = frame.sort_values(by="timeStamp")
            frame["rel_time"] = frame["timeStamp"] - frame["timeStamp"].min()

        if self.mode == "both":
            return extract_windows_aligned(frame, window_sec=self.window_sec)

        return extract_windows(frame, window_sec=self.window_sec)

In [90]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

model: RandomForestClassifier = joblib.load("../models/phone_train.pkl")
model_aligned: RandomForestClassifier = joblib.load("../models/both_train.pkl")
pipeline: Pipeline = joblib.load("../models/phone_train.pkl")

In [91]:
model_aligned

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('feature_engineering', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](18,)","['A','B','C',...,'Q','R','S']"
,mode,'both'
,window_sec,4
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",371
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",38
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1


In [78]:
pipeline_separated = Pipeline(
    [
        ("feature_extracting", window_extractor),
        ("classifier", model),
    ]
)

pipeline_aligned = Pipeline(
    [
        ("feature_extractor", window_extractor_aligned),
        ("classifier", model_aligned),
    ]
)

In [93]:
pipeline.predict(df_phone)

array(['G'], dtype=object)